# 05 — RDF-SYN-005 backend contract audit & offline red team
Project RDF-SFTW26-NEW-001. **Offline only**: local stores are `LOCAL_*_NOT_ATLAS`. No Atlas write, no deploy, no tunnel, no cloud telemetry. E2E = **NOT_TESTED**.

In [1]:
import sys, json, hashlib, subprocess, os, time, importlib.metadata as md_, datetime as dt
from pathlib import Path
ROOT = next(q for q in [Path.cwd(), *Path.cwd().parents] if (q/"integration/three_platform").exists()); assert (ROOT/"integration/three_platform/tools_contract.json").exists(), "run from repo root"
sys.path.insert(0, str(ROOT/"src")); NOW = lambda: dt.datetime.now(dt.timezone.utc).isoformat()
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
env = {"python": sys.version.split()[0], **{p: md_.version(p) for p in ("fastapi","pydantic","httpx","pytest","raindrop-ai","pymongo")}}
pins = {f: sha(ROOT/f) for f in ("integration/three_platform/tools_contract.json","integration/three_platform/atlas_ticket_fixtures.jsonl","integration/three_platform/raindrop_eval_dataset.csv","src/make_it_rain/backend/app.py","src/make_it_rain/backend/store.py")}
print(env); print(json.dumps(pins, indent=1))

{'python': '3.13.9', 'fastapi': '0.143.0', 'pydantic': '2.14.1', 'httpx': '0.28.1', 'pytest': '9.1.1', 'raindrop-ai': '0.0.75', 'pymongo': '4.18.3'}
{
 "integration/three_platform/tools_contract.json": "7550a6902ac66d77dd9b0d977a411de94c0ac2a9b95067a6607d03fa472cc7eb",
 "integration/three_platform/atlas_ticket_fixtures.jsonl": "e03beb1a5118bee5f8532939043b7fa69092cc527cf600cba68f109c4f952dbe",
 "integration/three_platform/raindrop_eval_dataset.csv": "2a2d7dd3c24b48989ce2d6a56c7f02aaa39210ad8417ffb3e78cab84930d4081",
 "src/make_it_rain/backend/app.py": "30b2f6f8e3669785ca6939cfe6116456ce08374339b6eddd15362e98939a4618",
 "src/make_it_rain/backend/store.py": "43200e03085572c6f0bb505888f951946cd7a54b078d2832b2c486f3988b238b"
}


In [2]:
EXPECT_FX = "e03beb1a5118bee5f8532939043b7fa69092cc527cf600cba68f109c4f952dbe"  # from atlas_readback_receipt.json
R = {}
R["fixture_hash"] = "PASS" if pins["integration/three_platform/atlas_ticket_fixtures.jsonl"] == EXPECT_FX else "FAIL"
fx = {json.loads(l)["_id"]: json.loads(l) for l in (ROOT/"integration/three_platform/atlas_ticket_fixtures.jsonl").read_text().splitlines()}
R["fixture_count_12"] = "PASS" if len(fx) == 12 else "FAIL"
R["syn005_expected"] = "PASS" if fx["RDF-SYN-005"]["expected_action"] == "SAFE_GUIDANCE" else "FAIL"
R["syn010_expected"] = "PASS" if fx["RDF-SYN-010"]["expected_action"] == "ESCALATE_STAFF" else "FAIL"
print(R)

{'fixture_hash': 'PASS', 'fixture_count_12': 'PASS', 'syn005_expected': 'PASS', 'syn010_expected': 'PASS'}


In [3]:
def run(paths):
    t = time.time()
    p = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", *paths], capture_output=True, text=True, cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT/"src"), "PYTHONDONTWRITEBYTECODE": "1"})
    return {"exit": p.returncode, "tail": p.stdout.strip().splitlines()[-1], "seconds": round(time.time()-t, 2)}
SUITES = {"backend": "tests/test_backend.py", "contract_conformance": "tests/test_contract_conformance.py", "redteam": "tests/test_redteam_backend.py", "shipped_receipt_checks": "integration/three_platform/test_receipt_checks.py"}
for k, v in SUITES.items():
    r = run([v]); R["suite_"+k] = ("PASS" if r["exit"] == 0 else "FAIL") + " " + r["tail"]; print(k, r)

backend {'exit': 0, 'tail': '15 passed, 1 warning in 0.34s', 'seconds': 0.74}


contract_conformance {'exit': 0, 'tail': '17 passed, 1 warning in 0.35s', 'seconds': 0.78}


redteam {'exit': 0, 'tail': '9 passed, 1 warning in 0.26s', 'seconds': 0.72}


shipped_receipt_checks {'exit': 0, 'tail': '5 passed in 0.02s', 'seconds': 0.35}


## RDF-SYN-005 offline flow: process A writes (local file store), fresh process B reads back independently

In [4]:
import tempfile
state = Path(tempfile.mkdtemp(prefix="rdf005_"))/"state.json"
A = r"""
import sys, json, os
sys.path.insert(0, sys.argv[1])
from fastapi.testclient import TestClient
from make_it_rain.backend.app import Config, create_app
from make_it_rain.backend.store import FileStore
fx=[json.loads(l) for l in open(sys.argv[1]+"/../integration/three_platform/atlas_ticket_fixtures.jsonl")]
c=TestClient(create_app(FileStore(sys.argv[2],fx),Config("k","s","agent_T")))
H={"x-rdf-tool-key":"k"}; b={"case_id":"RDF-SYN-005","idempotency_key":"idem-nb-0005"}
r1=c.post("/api/tools/create_test_ticket",headers=H,json=b).json(); r2=c.post("/api/tools/create_test_ticket",headers=H,json=b).json()
e=c.post("/api/tools/request_staff_escalation",headers=H,json={"ticket_id":r1["ticket_id"],"reason_code":"NEEDS_STAFF"}).json()
print(json.dumps({"pid":os.getpid(),"r1":r1,"r2":r2,"esc":e}))
"""
B = r"""
import sys, json, os, hashlib
d=json.load(open(sys.argv[1])); t=d["integration_smoke_tickets"]
print(json.dumps({"pid":os.getpid(),"ticket_count":len(t),"event_count":len(d["integration_smoke_events"]),"ticket":list(t.values())[0] if t else None,"fixture_count":len(d["ticket_fixtures"])}))
"""
pa = subprocess.run([sys.executable, "-c", A, str(ROOT/"src"), str(state)], capture_output=True, text=True, env={**os.environ, "PYTHONDONTWRITEBYTECODE": "1"})
pb = subprocess.run([sys.executable, "-c", B, str(state)], capture_output=True, text=True)
a = json.loads(pa.stdout.strip().splitlines()[-1]); b = json.loads(pb.stdout.strip().splitlines()[-1])
R["syn005_create_succeeded"] = "PASS" if a["r1"]["status"] == "SUCCEEDED" and a["r1"]["atlas_readback"] else "FAIL"
R["syn005_idempotent_one_ticket"] = "PASS" if a["r1"]["ticket_id"] == a["r2"]["ticket_id"] and b["ticket_count"] == 1 else "FAIL"
R["syn005_escalation_request_only"] = "PASS" if a["esc"] == {"status": "PENDING", "staff_confirmed": False} else "FAIL"
R["fresh_process_readback_LOCAL_FILE"] = "PASS" if a["pid"] != b["pid"] and b["ticket"]["state"] == "ESCALATION_REQUESTED" and b["ticket"]["case_id"] == "RDF-SYN-005" else "FAIL"
R["frozen_fixtures_intact_in_store"] = "PASS" if b["fixture_count"] == 12 else "FAIL"
print(a, b["pid"], b["ticket_count"], b["event_count"])

{'pid': 50882, 'r1': {'status': 'SUCCEEDED', 'ticket_id': 'SMOKE-2dbd0d4c063b6881', 'state': 'OPEN_TEST_ONLY', 'atlas_readback': True}, 'r2': {'status': 'SUCCEEDED', 'ticket_id': 'SMOKE-2dbd0d4c063b6881', 'state': 'OPEN_TEST_ONLY', 'atlas_readback': True}, 'esc': {'status': 'PENDING', 'staff_confirmed': False}} 50883 1 2


## Items that cannot run offline (explicit)

In [5]:
NOT_TESTED = {"atlas_credentialed_write_readback": "no app credentials in effective MONGODB_URI",
 "https_deployment": "no deploy/tunnel authorized", "elevenlabs_conversation_tool_call_webhook": "no public URL, webhook secret unset; agent id user-reported",
 "raindrop_cloud_trace": "no authorization/readback key; earlier local SDK emit used a placeholder conversation",
 "raindrop_judge_run": "not authorized", "hmac_vs_real_elevenlabs_delivery": "signature format implemented from docs; unverified against a real delivery",
 "worker_process_restart_on_atlas": "local file store only", "RDF-SYN-010_E2E": "same blockers"}
R.update({k: "NOT_TESTED" for k in NOT_TESTED}); R["e2e_RDF-SYN-005"] = "NOT_TESTED"
failed = [k for k, v in R.items() if v.startswith("FAIL")]
receipt = {"project_id": "RDF-SFTW26-NEW-001", "record_type": "OFFLINE_CONTRACT_AUDIT_RECEIPT", "observed_at": NOW(), "synthetic": True,
  "predecessor_commit": "5cf95a3edfe05e736655d960cb58f43f631fe401", "case_id": "RDF-SYN-005", "environment": env, "pinned_inputs": pins,
  "store_kind": "LOCAL_FILE_NOT_ATLAS", "results": R, "not_tested_reasons": NOT_TESTED, "failures": failed,
  "elevenlabs_conversation_id": None, "atlas_ticket_id": None, "raindrop_trace_id": None, "eval_run_id": None,
  "signed": "NOT_SIGNED", "mmr_root": "NOT_COMPUTED"}
out = ROOT/"results/three_platform/offline_audit_RDF-SYN-005_20261011.json"
out.write_text(json.dumps(receipt, indent=2)); print(json.dumps(R, indent=1)); print("FAILURES:", failed, "\nreceipt sha256", sha(out))
assert not failed

{
 "fixture_hash": "PASS",
 "fixture_count_12": "PASS",
 "syn005_expected": "PASS",
 "syn010_expected": "PASS",
 "suite_backend": "PASS 15 passed, 1 warning in 0.34s",
 "suite_contract_conformance": "PASS 17 passed, 1 warning in 0.35s",
 "suite_redteam": "PASS 9 passed, 1 warning in 0.26s",
 "suite_shipped_receipt_checks": "PASS 5 passed in 0.02s",
 "syn005_create_succeeded": "PASS",
 "syn005_idempotent_one_ticket": "PASS",
 "syn005_escalation_request_only": "PASS",
 "fresh_process_readback_LOCAL_FILE": "PASS",
 "frozen_fixtures_intact_in_store": "PASS",
 "atlas_credentialed_write_readback": "NOT_TESTED",
 "https_deployment": "NOT_TESTED",
 "elevenlabs_conversation_tool_call_webhook": "NOT_TESTED",
 "raindrop_cloud_trace": "NOT_TESTED",
 "raindrop_judge_run": "NOT_TESTED",
 "hmac_vs_real_elevenlabs_delivery": "NOT_TESTED",
 "worker_process_restart_on_atlas": "NOT_TESTED",
 "RDF-SYN-010_E2E": "NOT_TESTED",
 "e2e_RDF-SYN-005": "NOT_TESTED"
}
FAILURES: [] 
receipt sha256 26429159affac2737